# UECFoodPix Training and Validation Set Splitting

This Notebook accomplishes only one thing:

- Select **3000 training images** from the official 9000 training images, categorized by the main food categories;

- Select **500 validation images** from the remaining images;

- Save the results as a single JSON file for use by all models.


In [36]:
from pathlib import Path
from collections import defaultdict
import json
import random

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

ROOT = Path.cwd()

candidates = [ROOT / "UECFOODPIX" / "data",]

DATA_ROOT = next(
    (
        path
        for path in candidates
        if (path / "train9000.txt").is_file()
        and (path / "UECFoodPIX" / "train" / "mask").is_dir()
    ),
    None,
)

TRAIN_LIST = DATA_ROOT / "train9000.txt"
MASK_DIR = DATA_ROOT / "UECFoodPIX" / "train" / "mask"

TRAIN_SIZE = 3000
VAL_SIZE = 500
SEED = 9444
NUM_CLASSES = 103

OUTPUT_FILE = (
    ROOT
    / f"uecfoodpix_split_train{TRAIN_SIZE}_val{VAL_SIZE}.json"
)

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_FILE:", OUTPUT_FILE)


DATA_ROOT: f:\Class\1 unsw\COMP 9444 Neural Networks and Deep Learning\GP\Group Project\9444\UECFOODPIX\data
OUTPUT_FILE: f:\Class\1 unsw\COMP 9444 Neural Networks and Deep Learning\GP\Group Project\9444\uecfoodpix_split_train3000_val500.json


In [37]:
def dominant_class(mask_path: Path) -> int:
    """Returns the foreground category with the most pixels in the mask"""
    """background=0 is not included in the comparison."""
    mask = np.asarray(Image.open(mask_path))

    if mask.ndim == 3:
        mask = mask[..., 0]

    counts = np.bincount(
        mask.astype(np.int64).ravel(),
        minlength=NUM_CLASSES,
    )
    counts[0] = 0

    return int(counts.argmax())


def proportional_sample(ids, labels, sample_size, seed):
    """Sampling is performed according to the proportion of the categories in the candidate data, while ensuring the accuracy of the total sample size."""
    groups = defaultdict(list)

    for image_id in ids:
        groups[labels[image_id]].append(image_id)

    total = len(ids)
    raw_quota = {
        class_id: sample_size * len(group) / total
        for class_id, group in groups.items()
    }
    quota = {
        class_id: int(value)
        for class_id, value in raw_quota.items()
    }

    remaining = sample_size - sum(quota.values())

    order = sorted(
        groups,
        key=lambda class_id: (
            raw_quota[class_id] - quota[class_id],
            len(groups[class_id]),
            -class_id,
        ),
        reverse=True,
    )

    for class_id in order[:remaining]:
        quota[class_id] += 1

    rng = random.Random(seed)
    selected = []

    for class_id in sorted(groups):
        group = sorted(groups[class_id])
        rng.shuffle(group)
        selected.extend(group[:quota[class_id]])

    rng.shuffle(selected)
    return selected


In [38]:
all_ids = [
    line.strip()
    for line in TRAIN_LIST.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

labels = {
    image_id: dominant_class(MASK_DIR / f"{image_id}.png")
    for image_id in tqdm(all_ids, desc="Read masks")
}

train_ids = proportional_sample(
    ids=all_ids,
    labels=labels,
    sample_size=TRAIN_SIZE,
    seed=SEED,
)

train_set = set(train_ids)

remaining_ids = [
    image_id
    for image_id in all_ids
    if image_id not in train_set
]

val_ids = proportional_sample(
    ids=remaining_ids,
    labels=labels,
    sample_size=VAL_SIZE,
    seed=SEED + 1,
)

assert len(train_ids) == TRAIN_SIZE
assert len(val_ids) == VAL_SIZE
assert len(set(train_ids)) == TRAIN_SIZE
assert len(set(val_ids)) == VAL_SIZE
assert not (set(train_ids) & set(val_ids))

print("Num of train:", len(train_ids))
print("Num of val:", len(val_ids))
print("Number of repetitions:", len(set(train_ids) & set(val_ids)))


Read masks: 100%|██████████| 9000/9000 [00:17<00:00, 524.09it/s]

Num of train: 3000
Num of val: 500
Number of repetitions: 0


In [39]:
split_data = {
    "train_size": TRAIN_SIZE,
    "val_size": VAL_SIZE,
    "train_seed": SEED,
    "val_seed": SEED+1,
    "train_ids": train_ids,
    "val_ids": val_ids,
}

OUTPUT_FILE.write_text(
    json.dumps(split_data, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Created:", OUTPUT_FILE)


Created: f:\Class\1 unsw\COMP 9444 Neural Networks and Deep Learning\GP\Group Project\9444\uecfoodpix_split_train3000_val500.json


## How to use json in other notebook

```python
from pathlib import Path
import json

DATA_ROOT = Path.cwd() / "UECFOODPIX" / "data"

SPLIT_FILE = (
    DATA_ROOT
    / "uecfoodpix_split_train3000_val500.json"
)

split_data = json.loads(
    SPLIT_FILE.read_text(encoding="utf-8")
)

train_ids = split_data["train_ids"]
val_ids = split_data["val_ids"]

assert len(set(train_ids) & set(val_ids)) == 0
```
